# Statistical Baselining & Anomaly Detection

This notebook builds a **statistical baseline of benign behaviour** from the **CIC-UNSW-NB15** graph in Neo4j (loaded by [`loader.ipynb`](loader.ipynb)) and then uses that baseline to flag deviations as candidate anomalies. It is a simple, transparent counterpoint to machine learning approaches such as in [`ml-gds.ipynb`](ml-gds.ipynb) - no ML model, just **per-feature Z-scores** over groups defined by the graph schema.

This is a typical rules-based approach to anomaly detection, and is often used in practice as a first line of defence - it's easy to understand and implement, and can be effective at catching large deviations from normal behaviour. The graph structure allows us to define groups of flows that share certain characteristics (e.g., same source subnet, destination port, time of day) and build baselines for those groups.

> *What is a Z-score ?*
>
>A Z-score is a statistical measure that describes how many standard deviations a data point is from the mean of a distribution. It is calculated as:
>
>```Z = (X - μ) / σ```
>
>Where:
>- `X` is the value of the data point
>- `μ` is the mean of the distribution
>- `σ` is the standard deviation of the distribution

Two complementary baselines are built:

1. **Volume baseline** - hourly flow / byte counts per `(srcSubnet, dstPort, hourOfDay, dayOfWeek)`, used to spot **batches** of traffic that are louder than usual.
2. **Behavioural baseline** - mean & standard-deviation of `flowDuration` and `fwdPacketsPerSec` per `(dstPort, hourOfDay, dayOfWeek)`, used to spot **individual flows** that look unlike normal traffic to that service at that time.

The detectors are evaluated against a sample of known malicious flows pulled from the graph, giving a conversion rate ("how much of the labelled attack traffic does a 3-sigma baseline actually catch?") and a per-category breakdown.

## Setup

Let us connect to Neo4j through the `Neo4jAnalysis` helper.

In [1]:
import os
import asyncio
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from IPython.display import Image, display
from neo4j_analysis import Neo4jAnalysis

warnings.filterwarnings("ignore")
load_dotenv()

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titleweight"] = "bold"

analysis = Neo4jAnalysis(
    os.getenv("NEO4J_URI"),
    os.getenv("NEO4J_USERNAME"),
    os.getenv("NEO4J_PASSWORD"),
    os.getenv("NEO4J_DATABASE"),
)
print("connection ok:", analysis.verify_connection())

connection ok: True


## Building the volume baseline

The first baseline answers: **"How much traffic normally hits this destination port from this source subnet, at this hour of this day of the week?"**

We bucket benign flows by `(srcSubnet, dstPort, hourOfDay, dayOfWeek)` and aggregate flow counts and byte rates. Because the dataset spans a limited window, many buckets only contain a single observation - their `stDev` will be `0`, which the scorer protects against with a small epsilon further down. In production this baseline would be refreshed on a rolling window so the standard deviations reflect real variance.

In [2]:
network_volume_baseline = analysis.run_query_df("""
    // Group benign flows into hourly buckets per calendar date
MATCH (src:Host)-[:IN_SUBNET]->(srcSub:Subnet),
      (src)-[:SENDS]->(f:Flow)-[:ON_PORT]->(p:Port)
WHERE f.label = 'Benign' AND f.timestamp IS NOT NULL
WITH srcSub.cidr AS srcSubnet, 
     p.number AS dstPort, 
     f.timestamp.hour AS hourOfDay,
     f.timestamp.dayOfWeek AS dayOfWeek, 
     count(f) AS flowsInHour,
     sum(coalesce(f.flowBytesPerSec, 0)) AS bytesInHour

// Aggregate across all historical dates to compute mean and standard deviation
RETURN srcSubnet, 
       dstPort, 
       hourOfDay,
       dayOfWeek,
       avg(flowsInHour) AS historicalMeanFlows,
       stDev(flowsInHour) AS historicalStdFlows,
       avg(bytesInHour) AS historicalMeanBytes,
       stDev(bytesInHour) AS historicalStdBytes
    """)

display(network_volume_baseline.head())

,srcSubnet,dstPort,hourOfDay,dayOfWeek,historicalMeanFlows,historicalStdFlows,historicalMeanBytes,historicalStdBytes
0,59.166.0.0/24,80,7,4,2354.0,0.0,7.799243e+07,0.0
1,59.166.0.0/24,25740,7,4,118.0,0.0,7.234093e+06,0.0
2,59.166.0.0/24,6881,7,4,3647.0,0.0,2.965948e+08,0.0
3,59.166.0.0/24,38586,7,4,2.0,0.0,5.615926e+03,0.0
4,59.166.0.0/24,11922,7,4,118.0,0.0,2.274070e+07,0.0


## Building the behavioural baseline

The second baseline answers: **"What does a normal individual flow to this service at this time *look like*?"** - shape, not volume.

Rather than a single feature, we profile a **vector of per-flow features** (defined once in the `FEATURES` list) chosen to expose the families a duration-only detector misses:

- **timing / beaconing** - `flowIatMean`, `flowIatStd`, `idleMean`.
    - Regular inter-arrival gaps betray backdoors and worms that otherwise hide inside short, normal-looking connections.
- **packet-size profile** - `packetLengthMean`.
    - Uniform tiny packets signal scanning; oversized payloads signal exploits/shellcode.
- **direction asymmetry** - `downUpRatio`.
    - Heavy one-sided transfer separates exfiltration and probing from balanced sessions.
- **flag patterns** - `synFlagCount`, `rstFlagCount`.
    - Half-open and rejected connections are the canonical scan tells.
- plus the original `flowDuration` and `fwdPacketsPerSec`.

Three design choices keep this both meaningful and runnable on Aura:

1. **`protocol` joins the group key** - `(protocol, dstPortBucket, hourOfDay, dayOfWeek)`. TCP and UDP traffic to the same port behaves nothing alike, and pooling them inflated every bucket's spread.
2. **Robust statistics** - each feature is summarised by its **median** (centre) and **inter-quartile range** (`IQR = p75 − p25`, spread) instead of mean/standard-deviation. The IQR is divided by `1.349` at scoring time to recover a σ-equivalent, so the familiar "3-sigma" threshold still applies. A near-constant bucket now yields `IQR = 0`, which the scorer treats as *no signal* rather than dividing by a tiny epsilon and producing astronomical Z-scores.
3. **Percentiles are computed client-side** - to avoid aggregating large numbers of flows in the database, we pull a sample(set by `BENIGN_SAMPLE_FRACTION`) of benign flows for each bucket and compute the percentiles in Python.

> **Tuning:** `WELL_KNOWN_PORT_MAX` controls how many ports get an individual profile (the rest share one ephemeral bucket), and `BENIGN_SAMPLE_FRACTION` trades exactness for a faster pull.

In [3]:
# Per-flow "shape" features for the behavioural baseline.
# Picked to cover the blind spots of duration-only scoring:
#   - timing / beaconing .... flowIatMean, flowIatStd, idleMean
#   - packet-size profile ... packetLengthMean
#   - direction asymmetry ... downUpRatio
#   - flag patterns ......... synFlagCount, rstFlagCount  (half-open / rejected scans)
# This single list drives the baseline query, the scorer, and the evaluation pull, so
# adding a feature here is the only change needed to extend the detector.
FEATURES = [
    "flowDuration",  # connection length
    "fwdPacketsPerSec",  # forward packet rate
    "flowIatMean",  # mean inter-arrival time (beaconing cadence)
    "flowIatStd",  # IAT regularity (low std => machine-regular beacon)
    "idleMean",  # idle gaps between bursts
    "packetLengthMean",  # typical packet size
    "downUpRatio",  # download/upload asymmetry
    "synFlagCount",  # half-open connections (scanning)
    "rstFlagCount",  # rejected connections (scanning)
]

# The volume baseline (above) survives on Aura because count/avg/stDev are O(1)-per-group
# streaming aggregations. The behavioural baseline needs *percentiles*, and doing those
# server-side is the memory killer:
#   - percentileCont sorts every value in each group -> 27 blocking sorts that streamed
#     no rows and tripped Aura's read timeout.
#   - apoc.agg.statistics builds an HdrHistogram per feature per group; the heavy-tailed
#     features (flowIatMean, flowDuration, ...) span a huge dynamic range, so each
#     histogram allocates enormous bucket arrays
# So we stream the raw benign feature rows out once (a plain MATCH..RETURN streams
# continuously -> no blocking-aggregation timeout) and compute median / IQR in pandas,
# where 3.45M rows is ~360MB and groupby().quantile() is a few seconds. The percentile
# memory now lives on this machine, not on Aura's bounded heap.

# Ports below this are real services worth profiling individually (~96 of them, ~1.3M
# flows); the dataset records the responder's port as dstPort, so everything at/above it
# is an ephemeral/dynamic port whose per-port bucket is almost always a singleton
# (IQR = 0 -> no signal). Pooling them into one EPHEMERAL_PORT bucket cuts ~685k groups
# to ~490 and keeps the baseline DataFrame and the scorer's join small. The scorer (next
# cell) applies the identical rule to its join key so the two line up.
WELL_KNOWN_PORT_MAX = 1024  # raise toward 49152 (IANA dynamic boundary) to profile more
EPHEMERAL_PORT = -1  # registered service ports individually, at more memory.

# Optional speed knob: stream a random fraction of benign flows instead of all of them.
# 1.0 = exact (deterministic); lower (e.g. 0.25) trims the ~3.5M-row pull and its runtime
# proportionally while still giving robust percentiles for well-supported buckets.
BENIGN_SAMPLE_FRACTION = float(os.getenv("BENIGN_SAMPLE_FRACTION", 1.0))

_feature_cols = ", ".join(f"f.{x} AS {x}" for x in FEATURES)
_sample_clause = (
    "" if BENIGN_SAMPLE_FRACTION >= 1.0 else f"AND rand() < {BENIGN_SAMPLE_FRACTION}"
)

# Stream raw benign feature rows (no server-side aggregation). run_query_to_df uses the
# driver's native Result.to_df(), so we never build a 3.5M-element list of dicts.
benign_flows = analysis.run_query_to_df(f"""
    MATCH (f:Flow)-[:ON_PORT]->(p:Port)
    WHERE f.label = 'Benign' AND f.timestamp IS NOT NULL {_sample_clause}
    RETURN f.protocol            AS protocol,
           p.number              AS dstPort,
           f.timestamp.hour      AS hourOfDay,
           f.timestamp.dayOfWeek AS dayOfWeek,
           {_feature_cols}
    """)

# Pool ephemeral ports (see above). Same rule the scorer applies to its join key.
benign_flows["dstPortBucket"] = np.where(
    benign_flows["dstPort"] < WELL_KNOWN_PORT_MAX,
    benign_flows["dstPort"],
    EPHEMERAL_PORT,
)

# Robust centre (median) and spread (IQR = p75 - p25) per feature, per service bucket.
_keys = ["protocol", "dstPortBucket", "hourOfDay", "dayOfWeek"]
_grouped = benign_flows.groupby(_keys)[FEATURES]
_p25 = _grouped.quantile(0.25)
_p50 = _grouped.quantile(0.50)
_p75 = _grouped.quantile(0.75)

behavioural_baseline = (
    _p50.add_suffix("_median").join((_p75 - _p25).add_suffix("_iqr")).reset_index()
)

print(f"baseline buckets: {len(behavioural_baseline):,}  |  features: {len(FEATURES)}")
display(behavioural_baseline.head())

baseline buckets: 487  |  features: 9


,protocol,dstPortBucket,hourOfDay,dayOfWeek,flowDuration_median,fwdPacketsPerSec_median,flowIatMean_median,flowIatStd_median,idleMean_median,packetLengthMean_median,...,rstFlagCount_median,flowDuration_iqr,fwdPacketsPerSec_iqr,flowIatMean_iqr,flowIatStd_iqr,idleMean_iqr,packetLengthMean_iqr,downUpRatio_iqr,synFlagCount_iqr,rstFlagCount_iqr
0,6,-1,0,3,3469.0,2834.503640,177.000000,262.336616,0.0,0.0,...,0.0,35044.0,2004.075832,267.547619,793.479991,0.0,286.488372,1.0,4.0,0.0
1,6,-1,1,3,3498.0,2796.576990,180.500000,263.750829,0.0,0.0,...,0.0,40216.0,2148.715541,329.875000,938.544740,0.0,287.512195,1.0,4.0,0.0
2,6,-1,2,3,3488.0,2838.892832,176.500000,264.350480,0.0,0.0,...,0.0,37713.0,2211.186327,348.712636,984.607216,0.0,274.153846,1.0,4.0,0.0
3,6,-1,3,3,3591.0,2754.820937,182.833333,272.855858,0.0,0.0,...,0.0,41053.0,2361.817382,446.380125,1067.196588,0.0,281.744186,1.0,4.0,0.0
4,6,-1,4,3,3487.0,2739.726027,187.000000,268.382850,0.0,0.0,...,0.0,55084.0,2400.426320,502.963976,1295.187442,0.0,280.000000,1.0,4.0,0.0


## Scoring new traffic against the baselines

With the two baselines in hand, we define three helpers:

- `prepare_new_traffic` - unwraps Neo4j `DateTime` objects, extracts `hourOfDay` / `dayOfWeek`, and derives a `/24` subnet from the source IP when the graph query didn't already attach one. This is the join key the rest of the pipeline depends on.
- `score_behavioral_anomalies` - per-flow detector. Left-joins each incoming flow to the behavioural baseline on `(protocol, dstPort, hourOfDay, dayOfWeek)`, then computes a **robust Z-score for every feature** in `FEATURES` using the median and IQR (`z = (x − median) / (IQR / 1.349)`). A flow is flagged if **any** feature breaches `z_threshold`; the `breached_features` column records which ones, for triage. Where a bucket has `IQR = 0` the scale is undefined, so that feature's Z-score is left `NaN` rather than exploding - the deliberate fix for the runaway scores mean/σ produced. Flows with **no matching baseline entry** (a brand-new protocol/port/time combination) are also flagged - the absence of a profile is itself a signal.
- `score_volume_anomalies` - per-window detector. Aggregates the incoming traffic into the same `(srcSubnet, dstPort, hourOfDay, dayOfWeek)` buckets as the volume baseline, then Z-scores the observed flow and byte counts. Only **positive** deviations (more traffic than expected) are treated as anomalous - quiet periods aren't interesting here.

The behavioural detector divides by `IQR / 1.349` (the robust σ-equivalent) and guards the zero-IQR case; the volume detector keeps the simple mean/σ Z-score with a small `epsilon = 1e-6` to avoid divide-by-zero.

In [4]:
import numpy as np
import pandas as pd

# IQR-to-sigma constant: for a normal distribution IQR = 1.349 * sigma, so dividing the
# IQR by this recovers a robust standard-deviation estimate and keeps the "3 sigma"
# threshold interpretable across the mean/std and median/IQR worlds.
IQR_TO_SIGMA = 1.349


def prepare_new_traffic(df):
    """Extract temporal components and subnet from raw traffic data."""
    df = df.copy()

    # Step 1: Safely unpack Neo4j driver custom DateTime values to Python native datetimes
    if df["timestamp"].dtype == "object":
        df["timestamp"] = df["timestamp"].apply(
            lambda x: x.to_native() if hasattr(x, "to_native") else x
        )

    # Step 2: Now Pandas can safely parse them into native timestamps
    if not pd.api.types.is_datetime64_any_dtype(df["timestamp"]):
        df["timestamp"] = pd.to_datetime(df["timestamp"])

    df["hourOfDay"] = df["timestamp"].dt.hour
    df["dayOfWeek"] = df["timestamp"].dt.dayofweek + 1  # 1 = Monday, 7 = Sunday

    # Derive subnet if not already provided by the graph query
    if "srcSubnet" not in df.columns and "srcIp" in df.columns:
        df["srcSubnet"] = df["srcIp"].apply(
            lambda ip: ".".join(str(ip).split(".")[:3]) + ".0/24"
        )
    return df


def score_behavioral_anomalies(
    new_traffic_df, baseline_df, features=FEATURES, z_threshold=3.0
):
    """Per-flow detector: robust (median / IQR) Z-scores over a multi-feature service
    profile, keyed by (protocol, dstPortBucket, hourOfDay, dayOfWeek). A flow is flagged
    when *any* feature breaches the threshold, or when no baseline profile exists for its
    key."""
    traffic = prepare_new_traffic(new_traffic_df)

    # Pool ephemeral ports exactly as the baseline query did, so the join keys line up.
    # Raw dstPort is kept untouched for triage/reporting; dstPortBucket is join-only.
    traffic["dstPortBucket"] = np.where(
        traffic["dstPort"] < WELL_KNOWN_PORT_MAX, traffic["dstPort"], EPHEMERAL_PORT
    )

    # Join each flow to its service profile. Protocol is now part of the key.
    merged = pd.merge(
        traffic,
        baseline_df,
        on=["protocol", "dstPortBucket", "hourOfDay", "dayOfWeek"],
        how="left",
    )

    # No profile for this (protocol, port, hour, day) -> candidate anomaly in its own right.
    has_no_profile = merged[f"{features[0]}_median"].isna()

    z_cols = []
    for x in features:
        # Robust scale; IQR/1.349 ~= sigma. Where IQR == 0 (a degenerate, near-constant
        # bucket) the scale is 0 -> leave the Z-score NaN rather than exploding it.
        scale = merged[f"{x}_iqr"] / IQR_TO_SIGMA
        z = (merged[x] - merged[f"{x}_median"]) / scale.replace(0, np.nan)
        merged[f"z_{x}"] = z
        z_cols.append(f"z_{x}")

    abs_z = merged[z_cols].abs()
    breaches = (abs_z > z_threshold).any(axis=1)
    merged["max_absolute_z"] = abs_z.max(axis=1)
    # Human-readable list of which features tripped, for triage.
    merged["breached_features"] = (abs_z > z_threshold).apply(
        lambda row: ", ".join(c[2:] for c in z_cols if row[c]), axis=1
    )

    return merged[breaches | has_no_profile].copy()


def score_volume_anomalies(new_traffic_df, baseline_df, z_threshold=3.0):
    """Aggregates a batch of traffic and tests if the window volume is anomalous."""
    traffic = prepare_new_traffic(new_traffic_df)

    # Aggregate the target traffic batch to match the baseline format
    # Using inline lambda x: x.fillna(0).sum() safely handles any NaNs/Infs
    observed_volume = (
        traffic.groupby(["srcSubnet", "dstPort", "hourOfDay", "dayOfWeek"])
        .agg(
            observedFlows=("key", "count"),
            observedBytes=("flowBytesPerSec", lambda x: x.fillna(0).sum()),
        )
        .reset_index()
    )

    # Merge with the volume baseline
    merged = pd.merge(
        observed_volume,
        baseline_df,
        on=["srcSubnet", "dstPort", "hourOfDay", "dayOfWeek"],
        how="left",
    )

    has_no_profile = merged["historicalMeanFlows"].isna()

    # Calculate Volume Z-Scores
    epsilon = 1e-6
    merged["z_flows"] = (merged["observedFlows"] - merged["historicalMeanFlows"]) / (
        merged["historicalStdFlows"] + epsilon
    )
    merged["z_bytes"] = (merged["observedBytes"] - merged["historicalMeanBytes"]) / (
        merged["historicalStdBytes"] + epsilon
    )

    is_anomalous = (merged["z_flows"] > z_threshold) | (merged["z_bytes"] > z_threshold)

    return merged[is_anomalous | has_no_profile].copy()

## Evaluating against known malicious traffic

To test the detectors we don't need synthetic data - the graph already contains labelled attack traffic. We pull a 50k-flow sample of everything that isn't `Benign`, keeping the columns the scorers need (`key`, `srcIp`, `dstPort`, `timestamp`, `flowDuration`, `fwdPacketsPerSec`, `flowBytesPerSec`) plus the ground-truth `trueLabel` so we can break detection performance down by attack category.

In [5]:
# Pull a diverse sample of known malicious traffic across categories.
# We return protocol (now part of the join key), every behavioural FEATURE, plus
# flowBytesPerSec for the volume detector and the ground-truth label for the breakdown.
_feature_cols = ",\n           ".join(f"f.{x} AS {x}" for x in FEATURES)

malicious_traffic_df = analysis.run_query_df(f"""
    MATCH (src:Host)-[:SENDS]->(f:Flow)-[:ON_PORT]->(p:Port)
    WHERE f.label <> 'Benign' AND f.timestamp IS NOT NULL
    RETURN f.key AS key,
           src.ip AS srcIp,
           f.protocol AS protocol,
           p.number AS dstPort,
           f.timestamp AS timestamp,
           f.flowBytesPerSec AS flowBytesPerSec,
           {_feature_cols},
           f.label AS trueLabel
    LIMIT 50000
    """)

print(
    f"Successfully pulled {len(malicious_traffic_df)} known malicious flows for testing."
)
print("Sample of attack categories pulled:")
print(malicious_traffic_df["trueLabel"].value_counts())

Successfully pulled 50000 known malicious flows for testing.
Sample of attack categories pulled:
trueLabel
Exploits          17211
Fuzzers           16835
Reconnaissance     9352
Generic            2489
DoS                2458
Shellcode          1144
Backdoor            250
Worms               150
Analysis            111
Name: count, dtype: int64


## Running the detectors

We now score the malicious sample against both baselines at a `z_threshold` of 3.0 - the conventional "three sigma" boundary, where roughly 0.3% of normal observations would be flagged by chance alone.

In [6]:
# Run the detection algorithms using a 3.0 standard deviation threshold
detected_behavioral = score_behavioral_anomalies(
    malicious_traffic_df, behavioural_baseline, z_threshold=3.0
)

detected_volume = score_volume_anomalies(
    malicious_traffic_df, network_volume_baseline, z_threshold=3.0
)

## Detection performance

Compute the **conversion rate** (fraction of labelled malicious flows that the behavioural detector flagged), break it down by ground-truth attack category to see *which* attack families this baseline is good or bad at catching, and surface the most extreme deviations as a sanity check on the highest-confidence detections.

In [7]:
# Calculate what percentage of the known attacks were caught
total_attack_flows = len(malicious_traffic_df)
caught_behavioral = detected_behavioral["key"].nunique()

detection_rate = (caught_behavioral / total_attack_flows) * 100
print(f"--- Detection Summary ---")
print(
    f"Baseline caught {caught_behavioral}/{total_attack_flows} individual malicious flows ({detection_rate:.2f}% conversion rate)."
)
print(f"Identified {len(detected_volume)} anomalous subnet/port volume windows.")

# Breakdown detection performance by the actual attack category
if not detected_behavioral.empty:
    print("\n--- Detection Breakdowns by Ground-Truth Label ---")
    # Count how many of each attack category were flagged as anomalous
    print(detected_behavioral["trueLabel"].value_counts())

    print("\n--- Which features trip most often ---")
    # Each flagged flow lists the features that breached; tally them.
    print(
        detected_behavioral["breached_features"]
        .str.split(", ")
        .explode()
        .replace("", np.nan)
        .value_counts()
    )

    print("\n--- Top 5 Most Extreme Deviations ---")
    # Sort by the largest absolute robust Z-score across all features.
    display(
        detected_behavioral.sort_values(by="max_absolute_z", ascending=False)[
            [
                "trueLabel",
                "srcIp",
                "protocol",
                "dstPort",
                "breached_features",
                "max_absolute_z",
            ]
        ].head(5)
    )

--- Detection Summary ---
Baseline caught 26886/50000 individual malicious flows (53.77% conversion rate).
Identified 3476 anomalous subnet/port volume windows.

--- Detection Breakdowns by Ground-Truth Label ---
trueLabel
Exploits          9963
Reconnaissance    6521
Fuzzers           5637
Generic           1916
DoS               1432
Shellcode         1144
Backdoor           206
Worms               45
Analysis            22
Name: count, dtype: int64

--- Which features trip most often ---
breached_features
flowIatMean         13770
flowIatStd          12237
flowDuration         9852
packetLengthMean     7741
fwdPacketsPerSec     4749
downUpRatio           598
idleMean                3
synFlagCount            2
Name: count, dtype: int64

--- Top 5 Most Extreme Deviations ---


,trueLabel,srcIp,protocol,dstPort,breached_features,max_absolute_z
45152,Exploits,175.45.176.3,17,67,"flowDuration, fwdPacketsPerSec, flowIatMean, f...",8.428045e+07
49528,Fuzzers,175.45.176.3,17,520,"flowDuration, flowIatMean",3.121554e+07
6317,Fuzzers,175.45.176.2,17,520,"flowDuration, flowIatMean, packetLengthMean",2.831699e+07
6257,Fuzzers,175.45.176.2,17,520,"flowDuration, flowIatMean, packetLengthMean",2.727221e+07
14618,Fuzzers,175.45.176.2,17,514,"flowDuration, flowIatMean",2.666983e+07


## Wrap-up

**What the baseline catches (and what it misses)**
- The behavioural baseline scores a **vector of features** (timing, packet size, direction, TCP flags) rather than duration alone, so families that mimic short benign connections - `Worms`, `Backdoor`, `Analysis` - get more chances to trip a flag via their beaconing cadence or scan-like flag patterns.
- Detection is still **uneven across attack families**, and the `breached_features` tally shows *which* signal does the work for each one - a useful explainability mechanism over a single blended score.
- The volume detector contributes a separate, **batch-level** signal: subnet/port windows that look louder than the historical mean, even when the individual flows inside them don't.

**Why this baseline is a useful counterpart to Machine Learning approaches**
- It is **explainable** by construction: every alert points at a specific feature, a specific group key `(protocol, dstPort, hour, day)`, and a numeric robust Z-score.
- It needs **no model training and no labels** at inference time - just the benign aggregate from the graph.
- It is a **complement**, not a substitute, for the more sophisticated approach in [`ml-gds.ipynb`](ml-gds.ipynb): structural similarity catches the look-alikes this baseline can't, and statistical baselining catches the loud-and-weird outliers that look structurally normal.